In [ ]:
%pip install pandas numpy

In [10]:
import pandas as pd
import numpy as np


df = pd.read_csv("../../data/processed_hotel_bookings_v2.csv")


print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (78674, 36)


,LeadTime,IsRepeatedGuest,ReservedRoomType,Agent,month_1,month_2,month_3,month_4,month_5,month_6,...,has_previous_bookings_not_canceled,DepositType_Non Refund,DepositType_Refundable,CustomerType_Group,CustomerType_Transient,requirePark,special_req,ArrivalYear,ArrivalDay,ADR
0,6,0,0,100.263371,0,0,0,0,0,0,...,0,0,0,0,1,0,0,2015,1,0.0
1,88,0,0,123.151688,0,0,0,0,0,0,...,0,0,0,0,1,0,1,2015,1,76.5
2,65,0,0,123.151688,0,0,0,0,0,0,...,0,0,0,0,1,0,1,2015,1,68.0
3,92,0,0,123.151688,0,0,0,0,0,0,...,0,0,0,0,1,0,2,2015,1,76.5
4,100,0,0,123.151688,0,0,0,0,0,0,...,0,0,0,0,1,0,1,2015,2,76.5


In [ ]:
%pip install scikit-learn

In [11]:

from sklearn.model_selection import train_test_split

X = df.drop(columns=["ADR"])
y = df["ADR"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)


Training set: (62939, 35)
Test set: (15735, 35)


In [12]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

cross-validation setup

In [13]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

Define a shared cross-validation evaluation function

In [14]:
def evaluate_cv_model(name, model, X_train, y_train, cv):

    start = time.time()

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "mse": "neg_mean_squared_error",
            "rmse": "neg_root_mean_squared_error",
            "r2": "r2"
        },
        n_jobs=1
    )

    elapsed = time.time() - start

    return {
        "Model": name,
        "MAE": -scores["test_mae"].mean(),
        "MSE": -scores["test_mse"].mean(),
        "RMSE": -scores["test_rmse"].mean(),
        "R2": scores["test_r2"].mean(),
        "Cross-validation time (s)": elapsed
    }

baseline models


In [15]:
baseline_models = {
    "KNN": KNeighborsRegressor(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    )
}

In [16]:
print("Non-numeric columns:", 
      X_train.select_dtypes(exclude=["number"]).columns.tolist())

print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())

Non-numeric columns: []
Missing values in X_train: 0
Missing values in X_test: 0


Run cross-validation and collect the baseline results

In [17]:
baseline_results = []

for name, model in baseline_models.items():

    print(f"Evaluating {name}...")

    result = evaluate_cv_model(
        name,
        model,
        X_train,
        y_train,
        cv
    )

    baseline_results.append(result)

baseline_results_df = pd.DataFrame(baseline_results)

print(
    baseline_results_df
    .sort_values("RMSE")
    .to_string(index=False)
)

Evaluating KNN...
Evaluating Decision Tree...
Evaluating Random Forest...
        Model       MAE         MSE      RMSE       R2  Cross-validation time (s)
Random Forest  6.059373  621.076964 20.508052 0.775934                  13.642894
Decision Tree  7.339348  754.909872 24.288166 0.695629                   2.172143
          KNN 15.509294 1269.522145 34.163903 0.383378                   4.149150


PCA Pipeline
Each pipeline performs these steps in order:
1. Learn PCA components from the training portion of a fold.
2. Transform that fold's training features.
3. Train the regression model on the transformed features.
4. Transform the validation features using the same PCA transformation.
5. Evaluate the predictions on that validation fold.

In [18]:
pca_models = {
    "KNN + PCA": Pipeline([
        ("pca", PCA(
            n_components=0.95,
            svd_solver="full"
        )),
        ("model", KNeighborsRegressor())
    ]),

    "Random Forest + PCA": Pipeline([
        ("pca", PCA(
            n_components=0.95,
            svd_solver="full"
        )),
        ("model", RandomForestRegressor(
            random_state=42,
            n_jobs=-1
        ))
    ])
}

Train and evaluate the PCA models using cross-validation


In [19]:
pca_results = []

for name, model in pca_models.items():

    print(f"Evaluating {name}...")

    result = evaluate_cv_model(
        name,
        model,
        X_train,
        y_train,
        cv
    )

    pca_results.append(result)

pca_results_df = pd.DataFrame(pca_results)

print(
    pca_results_df
    .sort_values("RMSE")
    .to_string(index=False)
)

Evaluating KNN + PCA...
Evaluating Random Forest + PCA...
              Model       MAE        MSE      RMSE       R2  Cross-validation time (s)
          KNN + PCA 21.346601 1621.27818 39.175058 0.183072                   0.940956
Random Forest + PCA 21.577455 1788.98255 41.364477 0.080542                   8.732955


Find out how many components PCA retained

In [20]:
for name, model in pca_models.items():

    model.fit(X_train, y_train)

    pca_step = model.named_steps["pca"]

    print(f"\n{name}")
    print("Original feature count:", X_train.shape[1])
    print("Retained components:", pca_step.n_components_)
    print(
        "Explained variance:",
        round(pca_step.explained_variance_ratio_.sum(), 4)
    )


KNN + PCA
Original feature count: 35
Retained components: 1
Explained variance: 0.9679

Random Forest + PCA
Original feature count: 35
Retained components: 1
Explained variance: 0.9679


In [22]:
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np

# Fit PCA using the training features only
pca_check = PCA()
pca_check.fit(X_train)

# Show variance explained by each component
variance_df = pd.DataFrame({
    "Component": range(1, len(pca_check.explained_variance_ratio_) + 1),
    "Explained Variance": pca_check.explained_variance_ratio_,
    "Cumulative Variance": np.cumsum(pca_check.explained_variance_ratio_)
})

print(variance_df.head(15).to_string(index=False))

 Component  Explained Variance  Cumulative Variance
         1            0.967881             0.967881
         2            0.025553             0.993433
         3            0.005997             0.999430
         4            0.000261             0.999691
         5            0.000050             0.999741
         6            0.000039             0.999780
         7            0.000035             0.999815
         8            0.000026             0.999841
         9            0.000018             0.999859
        10            0.000017             0.999876
        11            0.000016             0.999891
        12            0.000013             0.999904
        13            0.000009             0.999913
        14            0.000009             0.999922
        15            0.000008             0.999930


In [23]:
feature_variance = X_train.var().sort_values(ascending=False)

print("Feature variances:")
print(feature_variance.to_string())

Feature variances:
LeadTime                              12262.647200
Agent                                   352.730722
ArrivalDay                               76.183731
TotalNights                               3.306124
market_segment_tier                       0.690116
special_req                               0.587875
ArrivalYear                               0.490200
total_guests                              0.398506
ReservedRoomType                          0.307893
WeekendOrNo                               0.249381
is_local                                  0.237787
CustomerType_Transient                    0.187925
Meal_BB                                   0.168996
DepositType_Non Refund                    0.136572
Meal_SC                                   0.115512
month_8                                   0.100354
month_5                                   0.092913
month_7                                   0.091512
month_6                                   0.090117
month_10    

In [24]:
loadings = pd.Series(
    pca_check.components_[0],
    index=X_train.columns
)

print("Features contributing most to PC1:")
print(loadings.abs().sort_values(ascending=False).head(15))

Features contributing most to PC1:
LeadTime                       0.998787
Agent                          0.049106
market_segment_tier            0.002340
DepositType_Non Refund         0.001417
ArrivalDay                     0.001059
special_req                    0.000946
is_local                       0.000883
ReservedRoomType               0.000730
CustomerType_Transient         0.000611
has_previous_cancellations     0.000502
TotalNights                    0.000500
Meal_SC                        0.000427
month_7                        0.000356
Meal_HB                        0.000338
month_2                        0.000288
dtype: float64


In [25]:
# Features with no variation
constant_columns = X_train.columns[X_train.nunique() <= 1]

print("Constant columns:")
print(list(constant_columns))

# Duplicate columns
duplicate_columns = X_train.columns[
    X_train.T.duplicated()
]

print("Duplicate columns:")
print(list(duplicate_columns))

Constant columns:
[]
Duplicate columns:
[]


In [26]:
scale_check = pd.DataFrame({
    "Mean": X_train.mean(),
    "Standard Deviation": X_train.std(),
    "Minimum": X_train.min(),
    "Maximum": X_train.max()
})

print(scale_check.to_string())

                                           Mean  Standard Deviation      Minimum     Maximum
LeadTime                             109.858593          110.736838     0.000000   629.00000
IsRepeatedGuest                        0.023451            0.151333     0.000000     1.00000
ReservedRoomType                       0.250735            0.554881     0.000000     3.00000
Agent                                105.906776           18.781127    37.549104   138.02899
month_1                                0.047204            0.212078     0.000000     1.00000
month_2                                0.062012            0.241180     0.000000     1.00000
month_3                                0.081936            0.274270     0.000000     1.00000
month_4                                0.094075            0.291936     0.000000     1.00000
month_5                                0.103656            0.304816     0.000000     1.00000
month_6                                0.100145            0.300195   

Create one comparison table

In [21]:
all_results_df = pd.concat(
    [baseline_results_df, pca_results_df],
    ignore_index=True
)

print(
    all_results_df[
        [
            "Model",
            "MAE",
            "MSE",
            "RMSE",
            "R2",
            "Cross-validation time (s)"
        ]
    ]
    .sort_values("RMSE")
    .to_string(index=False)
)

              Model       MAE         MSE      RMSE       R2  Cross-validation time (s)
      Random Forest  6.059373  621.076964 20.508052 0.775934                  13.642894
      Decision Tree  7.339348  754.909872 24.288166 0.695629                   2.172143
                KNN 15.509294 1269.522145 34.163903 0.383378                   4.149150
          KNN + PCA 21.346601 1621.278180 39.175058 0.183072                   0.940956
Random Forest + PCA 21.577455 1788.982550 41.364477 0.080542                   8.732955
